In [1]:
# HomeGuard Security System - Simulador de vigilancia domestica
# Autor: Jaime G. Marquerie
# Lee sensores (inventados con random), aplica reglas y muestra alertas.

import random
from datetime import datetime, timedelta

In [2]:
# --- PASO 2: estructuras de datos (diccionarios) ---

def create_sensor(sensor_id, location, sensor_type):
    """Crea la ficha (diccionario) de un sensor."""
    return {"id": sensor_id, "location": location,
            "type": sensor_type, "current_value": None}

def create_alert(level, message):
    """Crea la ficha (diccionario) de una alerta."""
    return {"level": level, "message": message}

def init_sensors():
    """Devuelve la lista con los 4 sensores de la casa."""
    return [
        create_sensor(1, "Living Room", "motion"),
        create_sensor(2, "Front", "door"),
        create_sensor(3, "Kitchen", "temperature"),
        create_sensor(4, "Bedroom", "smoke"),
    ]

print(init_sensors())   # comprobacion: deben salir 4 sensores

[{'id': 1, 'location': 'Living Room', 'type': 'motion', 'current_value': None}, {'id': 2, 'location': 'Front', 'type': 'door', 'current_value': None}, {'id': 3, 'location': 'Kitchen', 'type': 'temperature', 'current_value': None}, {'id': 4, 'location': 'Bedroom', 'type': 'smoke', 'current_value': None}]


In [3]:
# --- PASO 3: reglas (if / else). Devuelven una alerta o None ---

def check_reading(sensor_type, value, mode, door_minutes=0):
    """Decide si una lectura necesita alerta. Devuelve un diccionario o None."""
    # Seguridad: movimiento o puerta abierta con la casa vacia
    if sensor_type == "motion" and value and mode == "AWAY":
        return create_alert("HIGH", "SECURITY: Motion detected while in AWAY mode!")
    if sensor_type == "door" and value and mode == "AWAY":
        return create_alert("HIGH", "SECURITY: Front Door opened while in AWAY mode!")
    if sensor_type == "door" and value and door_minutes > 5:
        return create_alert("LOW", f"COMFORT: Door left open for {door_minutes} minutes")
    # Seguridad fisica: temperatura extrema y humo
    if sensor_type == "temperature":
        if value < 35:
            return create_alert("CRITICAL", f"SAFETY: {value}°F - frozen pipe risk!")
        if value > 95:
            return create_alert("CRITICAL", f"SAFETY: {value}°F - equipment failure risk!")
        if mode == "HOME" and (value < 65 or value > 75):
            return create_alert("LOW", f"COMFORT: {value}°F is outside 65-75°F")
    if sensor_type == "smoke" and value:
        return create_alert("CRITICAL", "SAFETY: Smoke detected - fire risk!")
    return None   # nada raro: no hay alerta

# Casos fijos (sabemos que deberia salir antes de ejecutar)
print(check_reading("temperature", 34, "HOME"))   # alerta CRITICAL
print(check_reading("temperature", 35, "HOME"))   # caso limite: comfort, no critical
print(check_reading("motion", True, "HOME"))      # None: estamos en casa
print(check_reading("motion", True, "AWAY"))      # alerta HIGH
print(check_reading("smoke", True, "HOME"))       # alerta CRITICAL

{'level': 'CRITICAL', 'message': 'SAFETY: 34°F - frozen pipe risk!'}
{'level': 'LOW', 'message': 'COMFORT: 35°F is outside 65-75°F'}
None
{'level': 'HIGH', 'message': 'SECURITY: Motion detected while in AWAY mode!'}
{'level': 'CRITICAL', 'message': 'SAFETY: Smoke detected - fire risk!'}


In [4]:
# --- PASOS 4 y 5: clase Sensor + funciones de mostrar y registrar ---

class Sensor:
    """Un sensor con sus datos (atributos) y sus acciones (metodos)."""

    def __init__(self, sensor_id, location, sensor_type):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.value = None
        self.door_minutes = 0          # minutos que la puerta lleva abierta

    def read(self):
        """Inventa una lectura realista segun el tipo de sensor."""
        if self.type == "temperature":
            self.value = random.randint(30, 100)
        elif self.type == "smoke":
            self.value = random.random() < 0.05      # 5 % de probabilidad
        else:                                        # motion o door
            self.value = random.random() < 0.3       # 30 % de probabilidad
        if self.type == "door":
            self.door_minutes = self.door_minutes + 1 if self.value else 0
        return self.value

    def isAbnormal(self, mode):
        """Devuelve la alerta de esta lectura, o None."""
        return check_reading(self.type, self.value, mode, self.door_minutes)

    def reset(self):
        """Vuelve el sensor a su estado inicial."""
        self.value = None
        self.door_minutes = 0

    def describe(self):
        """Texto de la lectura para mostrar en pantalla."""
        if self.type == "motion":
            return "Activity detected" if self.value else "No activity"
        if self.type == "door":
            return "OPENED" if self.value else "CLOSED"
        if self.type == "smoke":
            return "SMOKE!" if self.value else "CLEAR"
        return f"{self.value}°F"

def trigger_alert(alert, time_text):
    """Muestra la alerta y la registra."""
    print(f"[ALERT!] 🚨 {alert['level']}: {alert['message']}")
    log_event(time_text, "Sending notification to homeowner...")

event_log = []                          # aqui se guardan todos los eventos

def log_event(time_text, message):
    """Guarda un evento en el registro y lo muestra."""
    event_log.append(f"[{time_text}] {message}")
    print(f"[LOG] [{time_text}] {message}")

# Comprobacion de la clase
s = Sensor(3, "Kitchen", "temperature")
s.value = 20
print(s.describe(), s.isAbnormal("AWAY"))
s.reset()
print(s.value)

20°F {'level': 'CRITICAL', 'message': 'SAFETY: 20°F - frozen pipe risk!'}
None


In [5]:
# --- PASO 6: bucle principal del simulador ---

def run_simulation(mode, iterations=3, seed=None):
    """Lee todos los sensores 'iterations' veces (1 minuto cada vez)."""
    if seed is not None:
        random.seed(seed)               # misma semilla = mismos resultados
    sensors = [Sensor(s["id"], s["location"], s["type"]) for s in init_sensors()]
    clock = datetime(2026, 10, 27, 14, 30, 0)

    print("=== HomeGuard Security System ===")
    print(f"Mode: {mode}")
    for i in range(iterations):
        time_text = clock.strftime("%H:%M:%S")
        print(f"\nTime: {time_text}")
        security_alerts = 0
        for sensor in sensors:
            sensor.read()
            print(f"[READING] {sensor.location} {sensor.type.capitalize()}: {sensor.describe()}")
            alert = sensor.isAbnormal(mode)
            if alert:
                trigger_alert(alert, time_text)
                if alert["message"].startswith("SECURITY"):
                    security_alerts += 1
        # Varios sensores de seguridad a la vez = posible robo
        if security_alerts >= 2:
            trigger_alert(create_alert("CRITICAL", "Multiple sensors triggered - possible break-in!"), time_text)
        clock = clock + timedelta(minutes=1)
    print(f"\nTotal events logged: {len(event_log)}")

run_simulation("AWAY", iterations=3, seed=7)

=== HomeGuard Security System ===
Mode: AWAY

Time: 14:30:00
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:30:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 36°F
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 37°F
[READING] Bedroom Smoke: CLEAR

Time: 14:32:00
[READING] Living Room Motion: Activity detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected while in AWAY mode!
[LOG] [14:32:00] Sending notification to homeowner...
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:32:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 83°F
[READING] Bedroom Smoke: CLEAR
[ALERT!] 🚨 CRITICAL: Multiple sensors triggered - possible break-in!
[LOG] [14:32:00] Sending notification to homeowner...

Total ev

In [6]:
# --- PASO 7: pruebas de los 3 escenarios ---
print("\n##### Security test (AWAY) #####")
run_simulation("AWAY", iterations=3, seed=1)
print("\n##### Comfort test (HOME) #####")
run_simulation("HOME", iterations=3, seed=2)


##### Security test (AWAY) #####
=== HomeGuard Security System ===
Mode: AWAY

Time: 14:30:00
[READING] Living Room Motion: Activity detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected while in AWAY mode!
[LOG] [14:30:00] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 38°F
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 78°F
[READING] Bedroom Smoke: CLEAR

Time: 14:32:00
[READING] Living Room Motion: Activity detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected while in AWAY mode!
[LOG] [14:32:00] Sending notification to homeowner...
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:32:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 79°F
[READING] Bedroom Smoke: CLEAR
[ALERT!] 🚨 CRITICAL: Multiple sensors triggered - possible break-in!
[LOG] [14:32:00] Sending 